In [1]:
%%capture
%pip install -q "nltk>=3.9,<4" "spacy>=3.8,<4" "transformers>=5,<6"
%pip install matplotlib
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [51]:
import ipynbname
from pathlib import Path
ROOT_DIR = ipynbname.path().parent
ROOT_DIR = Path(ROOT_DIR)
print(ROOT_DIR)

from datasets import load_dataset
import pandas as pd
import re
from transformers import AutoTokenizer
xlm_tokeniser = AutoTokenizer.from_pretrained("xlm-roberta-base")

dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()
prime_train = df_train[(df_train["lang"].isin(['ar', 'ko', 'te']))]
prime_val   = df_val[(df_val["lang"].isin(['ar', 'ko', 'te']))]

/home/prga/Documents/uni/nlp/repos/msc-nlp-2026/project_notebooks


In [69]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def tokenize_wo_offsets(text):
    matches = list(TOKEN_PATTERN.finditer(text))
    tokens = [match.group(0) for match in matches]
    return tokens
    
def tokenize_with_offsets(text):
    matches = list(TOKEN_PATTERN.finditer(text))
    tokens = [match.group(0) for match in matches]
    offsets = [(match.start(), match.end()) for match in matches]
    return tokens, offsets

#def tokenize_with_offsets(text):
#    xlm_dict = xlm_tokeniser(text, return_offsets_mapping=True, truncation=True)
#    return xlm_tokeniser.convert_ids_to_tokens(xlm_dict["input_ids"]), xlm_dict["offset_mapping"]

def character_span_to_bio(context, answer_start=None, answer_text=""):
    tokens, offsets = tokenize_with_offsets(context)
    labels = ["O"] * len(tokens)

    if answer_start is None or answer_text == "":
        return tokens, offsets, labels

    answer_end = answer_start + len(answer_text)
    if context[answer_start:answer_end] != answer_text:
        raise ValueError("The supplied answer text does not match the character span")

    covered = [
        index
        for index, (start, end) in enumerate(offsets)
        if start < answer_end and end > answer_start
    ]
    if not covered:
        raise ValueError("The answer does not overlap any token")
    if offsets[covered[0]][0] != answer_start or offsets[covered[-1]][1] != (answer_end):
        #for index in covered:
            #print(f"{offsets[index]}")
        raise ValueError(f"The answer span {answer_start}:{answer_end} does not align with token boundaries {offsets[covered[0]][0]}:{offsets[covered[-1]][1]}")

    labels[covered[0]] = "B-ANS"
    for index in covered[1:]:
        labels[index] = "I-ANS"
    return tokens, offsets, labels


def bio_to_character_span(context, offsets, labels):
    if len(offsets) != len(labels):
        raise ValueError("Offsets and labels must have the same length")
    if not set(labels) <= {"O", "B-ANS", "I-ANS"}:
        raise ValueError("Only O, B-ANS and I-ANS labels are supported")

    answer_indices = [
        index for index, label in enumerate(labels) if label != "O"
    ]
    if not answer_indices:
        return None, ""

    start_index = answer_indices[0]
    expected_indices = list(range(start_index, start_index + len(answer_indices)))
    expected_labels = ["B-ANS"] + ["I-ANS"] * (len(answer_indices) - 1)
    if answer_indices != expected_indices:
        raise ValueError("The labels contain multiple or non-contiguous answer spans")
    if [labels[index] for index in answer_indices] != expected_labels:
        raise ValueError("The answer span must begin with B-ANS and continue with I-ANS")

    start = offsets[answer_indices[0]][0]
    end = offsets[answer_indices[-1]][1]
    return start, context[start:end]

In [53]:
ar_prime = prime_train[(prime_train["lang"] == 'ar')]
ko_prime = prime_train[(prime_train["lang"] == 'ko')]
te_prime = prime_train[(prime_train["lang"] == 'te')]

In [60]:
def test_bio(row):
    context = row['context']
    answer_start = row['answer_start']
    answer_text = row['answer']

    tokens, offsets, labels = character_span_to_bio(
        context, answer_start, answer_text
    )
    round_trip_start, round_trip_text = bio_to_character_span(
        context, offsets, labels
    )

    #print(f"Starts: {round_trip_start}:{answer_start}")
    #print(f"texts:\n{round_trip_text}\n{answer_text}")
    assert round_trip_start == answer_start
    assert round_trip_text == answer_text

    return pd.DataFrame({"token": tokens, "offset": offsets, "label": labels})
    

In [61]:
for index, row in ar_prime.head(5).iterrows():
    try:
        row_df = test_bio(row)
        #print(row_df[(row_df['label'] == 'B-ANS')])
        #print(row_df[(row_df['label'] == 'I-ANS')])
    except ValueError as e:
        print(e)

for index, row in ko_prime.head(5).iterrows():
    try:
        row_df = test_bio(row)
        #print(row_df[(row_df['label'] == 'B-ANS')])
        #print(row_df[(row_df['label'] == 'I-ANS')])
    except ValueError as e:
        print(e)

for index, row in te_prime.head(5).iterrows():
    try:
        row_df = test_bio(row)
        #print(row_df[(row_df['label'] == 'B-ANS')])
        #print(row_df[(row_df['label'] == 'I-ANS')])
    except ValueError as e:
        print(e)


The supplied answer text does not match the character span


In [186]:
import random
import re
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

import numpy as np
#import pandas as pd
random.seed(42)
seed = np.random.seed(42)

In [175]:
NER_TAGS = [
    "O", "B-ANS", "I-ANS",
]
TAG_TO_ID = {tag: index for index, tag in enumerate(NER_TAGS)}

In [176]:
def context_bio(row):
    context = row['context']
    answer_start = row['answer_start']
    answer_text = row['answer']
    try:
        tokens, offsets, labels = character_span_to_bio(
            context, answer_start, answer_text
        )
    except ValueError as e:
        tokens, offsets, labels = [], [], []
    return tokens, offsets, labels

In [177]:
sprime_train = prime_train.copy()
sprime_val = prime_val.copy()

In [178]:
token_series, offset_series, label_series = [], [], []
for index, row in prime_train.iterrows():
    tokens, offsets, labels = context_bio(row)
    token_series.append(tokens)
    offset_series.append(offsets)
    label_series.append(labels)
sprime_train['c_tokens'] = pd.Series(token_series)
sprime_train['c_offsets'] = pd.Series(offset_series)
sprime_train['c_labels'] = pd.Series(label_series)

token_series, offset_series, label_series = [], [], []
for index, row in prime_val.iterrows():
    tokens, offsets, labels = context_bio(row)
    token_series.append(tokens)
    offset_series.append(offsets)
    label_series.append(labels)
sprime_val['c_tokens'] = pd.Series(token_series)
sprime_val['c_offsets'] = pd.Series(offset_series)
sprime_val['c_labels'] = pd.Series(label_series)

sprime_train.head(1)

,question,context,lang,answerable,answer_start,answer,answer_inlang,c_tokens,c_offsets,c_labels
4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,None,"[The, airline, was, established, in, September...","[(0, 3), (4, 11), (12, 15), (16, 27), (28, 30)...","[O, O, O, O, O, B-ANS, I-ANS, O, O, O, O, O, O..."


In [179]:
datasets = {
    'train': sprime_train, 'validation': sprime_val
}

In [180]:
labels = []
for index, row in datasets['train'].iterrows():
    if type(row['c_labels']) != float:
        for label in row['c_labels']:
            labels.append(label)
label_counts = pd.Series(labels).value_counts()
label_counts


O        149155
I-ANS      3378
B-ANS      1482
Name: count, dtype: int64

In [181]:
pd.DataFrame({
    "label": NER_TAGS,
    "count": label_counts.values,
    "proportion": label_counts.values / label_counts.sum(),
})

,label,count,proportion
0,O,149155,0.968445
1,B-ANS,3378,0.021933
2,I-ANS,1482,0.009622


In [182]:
def bio_spans(sequence):
    spans = set()
    active_type = None
    start = None

    for index, label in enumerate(list(sequence) + ["O"]):
        #if (index % 1000) == 0:
        #    print(label)
        #label = NER_TAGS[int(label_id)]
        if label == "O":
            prefix, entity_type = "O", None
        else:
            #try:
            prefix, entity_type = label.split("-", 1)
            #except:
            #    print(label)

        if prefix == "I" and entity_type == active_type:
            continue

        if active_type is not None:
            spans.add((start, index, active_type))
            active_type = None
            start = None

        if prefix in {"B", "I"}:
            active_type = entity_type
            start = index

    return spans


def span_scores(gold_sequences, predicted_sequences):
    predicted_count = gold_count = matches = 0
    for gold, predicted in zip(gold_sequences, predicted_sequences):
        gold_spans = bio_spans(gold)
        predicted_spans = bio_spans(predicted)
        gold_count += len(gold_spans)
        predicted_count += len(predicted_spans)
        matches += len(gold_spans & predicted_spans)

    precision = matches / predicted_count if predicted_count else 0.0
    recall = matches / gold_count if gold_count else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def evaluate_sequences(name, gold_sequences, predicted_sequences):
    gold_flat = np.concatenate(gold_sequences)
    predicted_flat = np.concatenate(predicted_sequences)
    precision, recall, span_f1 = span_scores(gold_sequences, predicted_sequences)
    result = {
        "model": name,
        "token_accuracy": accuracy_score(gold_flat, predicted_flat),
        "entity_token_macro_f1": f1_score(
            gold_flat,
            predicted_flat,
            labels=list(range(1, len(NER_TAGS))),
            average="macro",
            zero_division=0,
        ),
        "span_precision": precision,
        "span_recall": recall,
        "span_f1": span_f1,
    }
    return result


def label_sequences(split):
    label_seq = []
    for index, row in datasets[split].iterrows():
        if type(row['c_labels']) != float:
            label_seq.append(row['c_labels'])
    return label_seq

In [183]:
validation_gold = label_sequences("validation")
only_o_predictions = [
    ["O"] * len(sequence) for sequence in validation_gold
]
baseline_scores = evaluate_sequences(
    "Always O", validation_gold, only_o_predictions
)
pd.DataFrame([baseline_scores])

,model,token_accuracy,entity_token_macro_f1,span_precision,span_recall,span_f1
0,Always O,0.971335,0.0,0.0,0.0,0.0


In [206]:
def token_features(tokens, index, question):
    word = tokens[index]
    features = {
        "bias": 1.0,
        "word.lower": word.lower(),
        "word.prefix2": word[:2].lower(),
        "word.suffix2": word[-2:].lower(),
        "word.suffix3": word[-3:].lower(),
        "word.istitle": word.istitle(),
        "word.isupper": word.isupper(),
        "word.isdigit": word.isdigit(),
        "contains_hyphen": "-" in word,
        #"question_length": len(question),
    }

    if index == 0:
        features["BOS"] = True
    else:
        previous = tokens[index - 1]
        features.update({
            "previous.lower": previous.lower(),
            "previous.istitle": previous.istitle(),
            "previous.isupper": previous.isupper(),
        })

    if index == len(tokens) - 1:
        features["EOS"] = True
    else:
        following = tokens[index + 1]
        features.update({
            "next.lower": following.lower(),
            "next.istitle": following.istitle(),
            "next.isupper": following.isupper(),
        })

    return features


def featurize_split(split, max_sentences=None):
    data = datasets[split]
    if max_sentences is not None:
        data = data.sample(min(max_sentences, len(data)), random_state=seed)

    features, labels, lengths = [], [], []
    for index, row in data.iterrows():
        if type(row['c_labels']) != float:
            tokens = row["c_tokens"]
            lengths.append(len(tokens))
            question = row["question"]
            features.extend(token_features(tokens, index, question) for index in range(len(tokens)))
            labels.extend(row["c_labels"])
    return features, np.asarray(labels), lengths


def split_by_lengths(values, lengths):
    sequences = []
    offset = 0
    for length in lengths:
        sequences.append(list(values[offset:offset + length]))
        offset += length
    assert offset == len(values)
    return sequences

In [207]:
TRAIN_SENTENCES = 4000

train_features, train_labels, train_lengths = featurize_split(
    "train", max_sentences=TRAIN_SENTENCES
)
validation_features, validation_labels, validation_lengths = featurize_split(
    "validation"
)

vectorizer = DictVectorizer(sparse=True)
X_train = vectorizer.fit_transform(train_features)
X_validation = vectorizer.transform(validation_features)
X_train.indices = X_train.indices.astype(np.int32) #added
X_train.indptr = X_train.indptr.astype(np.int32)   #added

token_classifier = SGDClassifier(
    loss="log_loss",
    alpha=1e-5,
    class_weight="balanced",
    max_iter=30,
    random_state=42,
)
token_classifier.fit(X_train, train_labels)

independent_flat = token_classifier.predict(X_validation)
independent_predictions = split_by_lengths(
    independent_flat, validation_lengths
)
independent_scores = evaluate_sequences(
    "Independent token classifier", validation_gold, independent_predictions
)

pd.DataFrame([baseline_scores, independent_scores])

/home/prga/Documents/uni/nlp/.venv/lib/python3.13/site-packages/sklearn/linear_model/_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


,model,token_accuracy,entity_token_macro_f1,span_precision,span_recall,span_f1
0,Always O,0.971335,0.0,0.00000,0.00000,0.000000
1,Independent token classifier,0.953886,0.0,0.04321,0.10728,0.061606
